# Video Deepfake Detector v3 — Colab Training

**What this does:** Fine-tunes a CLIP ViT-B/16 video deepfake detector for BitMind GAS (Subnet 34).

**Key v3 fixes** (vs v2 which failed entrance exam at 77%):
- First 16 consecutive frames (matches gasbench/decord eval pipeline)
- REAL_CLASS_WEIGHT=2.0 (fixes false positives on real videos)
- Aspect-preserving resize + center crop

**Instructions:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells (Runtime → Run all)
3. Download the output zip when done
4. Push from your Ubuntu machine:
```bash
cd ~/bitmind-subnet && source .venv/bin/activate
python push_standalone.py --video-model /path/to/video_detector_v3.zip --wallet-name miner
```

## Step 1: Check GPU & Install Dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
!pip install -q timm safetensors

## Step 2: Download Existing Model Weights from GitHub

In [ ]:
import os
os.makedirs('/content/pretrained', exist_ok=True)

# Clone repo with LFS to get model weights
!git lfs install
!git clone https://github.com/hochul325/deepfake-detector.git /content/repo 2>/dev/null || (cd /content/repo && git pull)

# Check if weights downloaded properly (should be ~344MB, not a pointer file)
import os
weights_path = '/content/repo/video_detector/model.safetensors'
size_mb = os.path.getsize(weights_path) / 1e6
print(f'Video model weights: {size_mb:.1f} MB')
if size_mb < 10:
    print('WARNING: Weights file is too small — LFS pointer not resolved!')
    print('Trying manual LFS pull...')
    !cd /content/repo && git lfs pull
    size_mb = os.path.getsize(weights_path) / 1e6
    print(f'After LFS pull: {size_mb:.1f} MB')
else:
    print('Weights loaded successfully')

## Step 3: Install gasbench & Download Video Datasets

This downloads the video evaluation datasets from BitMind. Takes ~10-20 min.

In [ ]:
# Install gasbench
!pip install -q gasbench 2>/dev/null || pip install -q "gasbench @ git+https://github.com/BitMind-AI/gasbench.git@main" 2>/dev/null

# Check if gasbench installed
try:
    import gasbench
    print(f'gasbench installed: {gasbench.__version__ if hasattr(gasbench, "__version__") else "OK"}')
except ImportError:
    print('gasbench import failed — will try CLI directly')

In [ ]:
# Download video datasets via gasbench
# This downloads real + synthetic video datasets used for GAS evaluation
GASBENCH_CACHE = '/content/gasbench_cache'
os.makedirs(GASBENCH_CACHE, exist_ok=True)

!gasbench download --modality video --output-dir {GASBENCH_CACHE} --small 2>&1 || echo 'gasbench download failed, trying alternative...'

# Check what we got
if os.path.exists(GASBENCH_CACHE):
    datasets = [d for d in os.listdir(GASBENCH_CACHE) if os.path.isdir(os.path.join(GASBENCH_CACHE, d))]
    print(f'Downloaded {len(datasets)} datasets')
    for d in sorted(datasets):
        info_path = os.path.join(GASBENCH_CACHE, d, 'dataset_info.json')
        if os.path.exists(info_path):
            import json
            info = json.load(open(info_path))
            print(f'  {d}: {info.get("modality", "?")} / {info.get("media_type", "?")}')

## Step 3b: Alternative — Manual Dataset Download

If gasbench download doesn't work, run this cell to download datasets directly from HuggingFace.

**Skip this cell if Step 3 worked.**

In [ ]:
# SKIP if gasbench download worked above!
# This manually downloads some common video deepfake datasets

SKIP_MANUAL = True  # Set to False if you need this

if not SKIP_MANUAL:
    !pip install -q datasets huggingface_hub
    from huggingface_hub import HfApi
    api = HfApi()

    # List bitmind datasets
    datasets = api.list_datasets(author='bitmind', search='video')
    for ds in datasets:
        print(f'  {ds.id}')

    print('\nYou may need to download specific datasets manually.')
    print('Check: https://huggingface.co/bitmind for available datasets')

## Step 4: Training Configuration

In [ ]:
# === CONFIGURATION ===
# Adjust these as needed

PRETRAINED_WEIGHTS = '/content/repo/video_detector/model.safetensors'
OUTPUT_DIR = '/content/video_detector_v3'
FRAME_DIR = '/content/video_frames_cache'

# v3 KEY CHANGES
FRAMES_PER_VIDEO = 16       # Was 8 in v2
REAL_CLASS_WEIGHT = 2.0     # Was 1.1 in v2

SEED = 42
EPOCHS = 35
BATCH_SIZE = 16  # For T4 16GB VRAM
LR = 3e-6
VAL_SPLIT = 0.20

import random, logging, sys
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s',
                    stream=sys.stdout, force=True)
logger = logging.getLogger(__name__)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Step 5: Model & Data Pipeline

In [ ]:
import io, json, zipfile
from pathlib import Path
from collections import Counter
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
import timm
from safetensors.torch import save_file, load_file
import cv2


# === Frame Extraction — matches gasbench/decord pipeline ===
def extract_frames_consecutive(video_path, num_frames=FRAMES_PER_VIDEO):
    """Extract first N consecutive frames with aspect-preserving resize + center crop."""
    frames = []
    try:
        cap = cv2.VideoCapture(str(video_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            return frames
        for i in range(min(num_frames, total)):
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h, w = frame.shape[:2]
            if h < w:
                new_h, new_w = 256, int(w * 256 / h)
            else:
                new_h, new_w = int(h * 256 / w), 256
            frame = cv2.resize(frame, (new_w, new_h), interpolation=cv2.INTER_AREA)
            y = (new_h - 256) // 2
            x = (new_w - 256) // 2
            frame = frame[y:y+256, x:x+256]
            frames.append(frame)
        cap.release()
    except Exception as e:
        logger.warning(f'Failed: {video_path}: {e}')
    return frames


def extract_all_frames_to_disk(videos, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / 'manifest.json'
    if manifest_path.exists():
        with open(manifest_path) as f:
            manifest = json.load(f)
        if len(manifest) > 0 and os.path.exists(manifest[0]['path']):
            logger.info(f'Cached: {len(manifest)} frames')
            return [(m['path'], m['label']) for m in manifest]
    manifest = []
    frame_idx = 0
    for i, (vpath, label, ds_name) in enumerate(videos):
        frames = extract_frames_consecutive(vpath)
        for j, frame in enumerate(frames):
            fname = f'frame_{frame_idx:06d}.jpg'
            fpath = str(output_dir / fname)
            Image.fromarray(frame).save(fpath, quality=95)
            manifest.append({'path': fpath, 'label': label})
            frame_idx += 1
        if (i + 1) % 100 == 0:
            logger.info(f'  Extracted {i+1}/{len(videos)} ({frame_idx} frames)')
    logger.info(f'  Total: {frame_idx} frames from {len(videos)} videos')
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f)
    return [(m['path'], m['label']) for m in manifest]


# === Dataset Discovery ===
def load_gasbench_video_datasets():
    cache = Path(GASBENCH_CACHE)
    all_videos = []
    for ds_dir in sorted(cache.iterdir()):
        if not ds_dir.is_dir():
            continue
        info_path = ds_dir / 'dataset_info.json'
        week_dirs = [d for d in ds_dir.iterdir() if d.is_dir() and d.name.startswith('20')]
        if week_dirs:
            for week_dir in week_dirs:
                info_path = week_dir / 'dataset_info.json'
                if not info_path.exists(): continue
                with open(info_path) as f: info = json.load(f)
                if info.get('modality') != 'video': continue
                label = 0 if info.get('media_type', '') == 'real' else 1
                ds_name = f'{ds_dir.name}/{week_dir.name}'
                samples_dir = week_dir / 'samples'
                if samples_dir.exists():
                    count = 0
                    for vid in sorted(samples_dir.iterdir()):
                        if vid.suffix.lower() in ('.mp4', '.avi', '.webm', '.mov'):
                            all_videos.append((str(vid), label, ds_name))
                            count += 1
                    print(f'  {ds_name}: {count} {"real" if label==0 else "synth"}')
            continue
        if not info_path.exists(): continue
        with open(info_path) as f: info = json.load(f)
        if info.get('modality') != 'video': continue
        label = 0 if info.get('media_type', '') == 'real' else 1
        ds_name = info.get('name', ds_dir.name)
        samples_dir = ds_dir / 'samples'
        if not samples_dir.exists(): continue
        count = 0
        for vid in sorted(samples_dir.iterdir()):
            if vid.suffix.lower() in ('.mp4', '.avi', '.webm', '.mov'):
                all_videos.append((str(vid), label, ds_name))
                count += 1
        print(f'  {ds_name}: {count} {"real" if label==0 else "synth"}')
    return all_videos


# === Model ===
class VideoDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2, backbone='vit_base_patch16_clip_224.openai'):
        super().__init__()
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.encoder = timm.create_model(backbone, pretrained=True, num_classes=0)
        feature_dim = self.encoder.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim), nn.Dropout(0.3),
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self.register_buffer('temperature', torch.ones(1))

    def extract_features(self, x):
        x = x / 255.0
        x = (x - self.mean) / self.std
        return self.encoder(x)

    def forward(self, x):
        if x.dim() == 4:
            features = self.extract_features(x)
            return self.classifier(features) / self.temperature
        B, T = x.shape[:2]
        frames = x.reshape(B * T, *x.shape[2:])
        features = self.extract_features(frames)
        features = features.view(B, T, -1)
        pooled = features.mean(dim=1)
        return self.classifier(pooled) / self.temperature


# === Augmentations ===
class JPEGCompress:
    def __init__(self, qr=(30, 95)): self.qr = qr
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=random.randint(*self.qr))
        buf.seek(0)
        return Image.open(buf).convert('RGB')

class RandomNoise:
    def __init__(self, sr=(0.005, 0.03)): self.sr = sr
    def __call__(self, img):
        arr = np.array(img, dtype=np.float32) / 255.0
        arr = np.clip(arr + np.random.randn(*arr.shape).astype(np.float32) * random.uniform(*self.sr), 0, 1)
        return Image.fromarray((arr * 255).astype(np.uint8))

class RandomDownscale:
    def __init__(self, sr=(0.4, 0.8)): self.sr = sr
    def __call__(self, img):
        w, h = img.size
        s = random.uniform(*self.sr)
        img = img.resize((max(16, int(w*s)), max(16, int(h*s))), Image.BILINEAR)
        return img.resize((w, h), Image.BILINEAR)


def get_train_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size + 16, size + 16)),
        transforms.RandomCrop((size, size)),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.20, hue=0.05),
        transforms.RandomApply([JPEGCompress()], p=0.3),
        transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.15),
        transforms.RandomApply([RandomNoise()], p=0.15),
        transforms.RandomApply([RandomDownscale()], p=0.15),
        transforms.RandomGrayscale(p=0.03),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.10, scale=(0.02, 0.10)),
        transforms.Lambda(lambda x: x * 255.0),
    ])

def get_val_transform(size=224):
    return transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x * 255.0),
    ])


class FrameDataset(Dataset):
    def __init__(self, fl, transform=None):
        self.fl = fl
        self.transform = transform
    def __len__(self): return len(self.fl)
    def __getitem__(self, idx):
        path, label = self.fl[idx]
        try: img = Image.open(path).convert('RGB')
        except: img = Image.new('RGB', (256, 256), (128, 128, 128))
        if self.transform: img = self.transform(img)
        return img, label

print('All classes defined.')

## Step 6: Load Data & Prepare Training

In [ ]:
# Discover datasets
print('Discovering video datasets...')
all_videos = load_gasbench_video_datasets()

n_real = sum(1 for _, l, _ in all_videos if l == 0)
n_synth = sum(1 for _, l, _ in all_videos if l == 1)
print(f'\nTotal: {len(all_videos)} videos ({n_real} real, {n_synth} synth)')

if len(all_videos) == 0:
    print('\nERROR: No videos found!')
    print(f'Check GASBENCH_CACHE = {GASBENCH_CACHE}')
    print('Make sure gasbench download worked in Step 3')
else:
    # Oversample small datasets
    ds_counts = Counter(ds for _, _, ds in all_videos)
    median_count = max(sorted(ds_counts.values())[len(ds_counts) // 2], 50)
    balanced = []
    for ds_name, count in ds_counts.items():
        items = [(p, l, d) for p, l, d in all_videos if d == ds_name]
        if count < median_count:
            items = items * max(1, median_count // count)
        balanced.extend(items)
    all_videos = balanced
    n_real = sum(1 for _, l, _ in all_videos if l == 0)
    n_synth = sum(1 for _, l, _ in all_videos if l == 1)
    print(f'After oversampling: {len(all_videos)} ({n_real} real, {n_synth} synth)')

    # Split
    random.shuffle(all_videos)
    val_size = int(len(all_videos) * VAL_SPLIT)
    val_videos = all_videos[:val_size]
    train_videos = all_videos[val_size:]
    print(f'Train: {len(train_videos)}, Val: {len(val_videos)}')

In [ ]:
# Extract frames to disk
print('Extracting train frames (first 16 consecutive)...')
train_frames = extract_all_frames_to_disk(train_videos, os.path.join(FRAME_DIR, 'train'))
print('Extracting val frames...')
val_frames = extract_all_frames_to_disk(val_videos, os.path.join(FRAME_DIR, 'val'))
print(f'Train: {len(train_frames)} frames, Val: {len(val_frames)} frames')

In [ ]:
# Create dataloaders
train_ds = FrameDataset(train_frames, get_train_transform())
val_ds = FrameDataset(val_frames, get_val_transform())

train_labels = [l for _, l in train_frames]
cc = [train_labels.count(0), train_labels.count(1)]
print(f'Class counts: real={cc[0]}, synth={cc[1]}')

if min(cc) > 0:
    weights = [1.0 / cc[l] for l in train_labels]
    sampler = WeightedRandomSampler(weights, len(train_labels), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=2, pin_memory=True, drop_last=True)
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)

val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

## Step 7: Create Model & Load Pretrained Weights

In [ ]:
model = VideoDeepfakeDetector().to(device)

if os.path.exists(PRETRAINED_WEIGHTS):
    print(f'Loading pretrained weights...')
    state = load_file(PRETRAINED_WEIGHTS)
    state = {k: v for k, v in state.items() if k != 'temperature'}
    model.load_state_dict(state, strict=False)
    print('Loaded!')
else:
    print('No pretrained weights — training from CLIP ViT scratch')

for p in model.parameters(): p.requires_grad = True
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

## Step 8: Train!

In [ ]:
# Training functions
def train_epoch(model, loader, optimizer, scheduler, device, epoch, mixup_alpha=0.2):
    model.train()
    weight = torch.tensor([REAL_CLASS_WEIGHT, 1.0]).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.08, weight=weight)
    total_loss, correct, total = 0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        if mixup_alpha > 0 and random.random() < 0.3:
            lam = max(l := np.random.beta(mixup_alpha, mixup_alpha), 1 - l)
            idx = torch.randperm(images.size(0), device=device)
            logits = model(lam * images + (1 - lam) * images[idx])
            loss = lam * criterion(logits, labels) + (1 - lam) * criterion(logits, labels[idx])
        else:
            logits = model(images)
            loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
        if (batch_idx + 1) % 50 == 0:
            print(f'Ep{epoch} [{batch_idx+1}/{len(loader)}] Loss:{total_loss/(batch_idx+1):.4f} Acc:{100*correct/total:.1f}%')
    return total_loss / max(len(loader), 1), correct / max(total, 1)

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    for images, labels in loader:
        logits = model(images.to(device))
        probs = F.softmax(logits, dim=1)
        all_labels.extend(labels.numpy())
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    labels = np.array(all_labels)
    preds = np.array(all_preds)
    probs = np.array(all_probs)
    acc = (preds == labels).mean()
    tp = ((preds==1)&(labels==1)).sum(); tn = ((preds==0)&(labels==0)).sum()
    fp = ((preds==1)&(labels==0)).sum(); fn = ((preds==0)&(labels==1)).sum()
    denom = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
    mcc = float(tp*tn - fp*fn) / denom if denom > 0 else 0
    brier = float(np.mean((probs[:, 1] - labels) ** 2))
    mcc_norm = max(0, ((mcc + 1) / 2)) ** 1.2
    brier_norm = max(0, (0.25 - brier) / 0.25) ** 1.8
    sn34 = float(np.sqrt(max(1e-12, mcc_norm * brier_norm)))
    print(f'  Acc:{acc:.4f} MCC:{mcc:.4f} Brier:{brier:.6f} sn34:{sn34:.4f} (TP={tp} TN={tn} FP={fp} FN={fn})')
    return {'accuracy': acc, 'mcc': mcc, 'brier': brier, 'sn34_score': sn34}

print('Training functions ready.')

In [ ]:
# === TRAINING LOOP ===
backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' not in n]
classifier_params = [p for n, p in model.named_parameters() if p.requires_grad and 'classifier' in n]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': LR},
    {'params': classifier_params, 'lr': LR * 10},
], weight_decay=0.02)

total_steps = len(train_loader) * EPOCHS
warmup_steps = len(train_loader) * 2

def lr_lambda(step):
    if step < warmup_steps: return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_sn34, best_epoch = 0, 0
patience, patience_counter = 10, 0

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*50}')
    print(f'Epoch {epoch}/{EPOCHS}')
    print(f'{"="*50}')

    loss, acc = train_epoch(model, train_loader, optimizer, scheduler, device, epoch)
    print(f'Train Loss:{loss:.4f} Acc:{acc:.4f}')

    metrics = evaluate(model, val_loader, device)

    if metrics['sn34_score'] > best_sn34:
        best_sn34 = metrics['sn34_score']
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), '/content/best_video_v3.pt')
        print(f'  *** New best sn34: {best_sn34:.4f} ***')
    else:
        patience_counter += 1
        if patience_counter >= patience and epoch > 5:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest epoch: {best_epoch} (sn34: {best_sn34:.4f})')

## Step 9: Calibrate & Package

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('/content/best_video_v3.pt', weights_only=True))

# Calibrate temperature
def calibrate_temperature(model, loader, device):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        old_temp = model.temperature.clone()
        model.temperature.fill_(1.0)
        for images, labels in loader:
            all_logits.append(model(images.to(device)).cpu())
            all_labels.append(labels)
        model.temperature.copy_(old_temp)
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    best_sn34, best_temp = 0, 1.0
    for temp in np.arange(0.1, 8.0, 0.02):
        probs = F.softmax(logits / temp, dim=1)
        sp = probs[:, 1]
        brier = ((sp - labels.float()) ** 2).mean().item()
        preds = (sp > 0.5).long()
        tp=((preds==1)&(labels==1)).sum().item(); tn=((preds==0)&(labels==0)).sum().item()
        fp=((preds==1)&(labels==0)).sum().item(); fn=((preds==0)&(labels==1)).sum().item()
        d = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
        mcc = float(tp*tn - fp*fn) / d if d > 0 else 0
        mn = max(0, ((mcc+1)/2))**1.2
        bn = max(0, (0.25-brier)/0.25)**1.8
        s = float(np.sqrt(max(1e-12, mn*bn)))
        if s > best_sn34: best_sn34, best_temp = s, temp
    for temp in np.arange(max(0.01, best_temp-0.15), best_temp+0.15, 0.001):
        probs = F.softmax(logits / temp, dim=1)
        sp = probs[:, 1]
        brier = ((sp - labels.float()) ** 2).mean().item()
        preds = (sp > 0.5).long()
        tp=((preds==1)&(labels==1)).sum().item(); tn=((preds==0)&(labels==0)).sum().item()
        fp=((preds==1)&(labels==0)).sum().item(); fn=((preds==0)&(labels==1)).sum().item()
        d = np.sqrt(float((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)))
        mcc = float(tp*tn - fp*fn) / d if d > 0 else 0
        mn = max(0, ((mcc+1)/2))**1.2
        bn = max(0, (0.25-brier)/0.25)**1.8
        s = float(np.sqrt(max(1e-12, mn*bn)))
        if s > best_sn34: best_sn34, best_temp = s, temp
    print(f'Calibrated: temp={best_temp:.3f} sn34={best_sn34:.4f}')
    model.temperature.fill_(best_temp)

calibrate_temperature(model, val_loader, device)
print('\nFinal evaluation:')
final = evaluate(model, val_loader, device)

In [ ]:
# Package model as GAS-ready zip
out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
save_file(model.state_dict(), str(out / 'model.safetensors'))

(out / 'model_config.yaml').write_text("""name: "clip-vit-video-deepfake-detector-gas"
version: "12.0.0"
modality: "video"

preprocessing:
  resize: [224, 224]
  max_frames: 16

model:
  num_classes: 2
  weights_file: "model.safetensors"
""")

(out / 'model.py').write_text('''import torch
import torch.nn as nn
import timm
from safetensors.torch import load_file


class VideoDeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        self.encoder = timm.create_model("vit_base_patch16_clip_224.openai", pretrained=False, num_classes=0)
        feature_dim = self.encoder.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(feature_dim), nn.Dropout(0.3),
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        self.register_buffer("temperature", torch.ones(1))

    def forward(self, x):
        if x.dim() == 4:
            x = x.unsqueeze(1)
        B, T = x.shape[:2]
        frames = x.reshape(B * T, *x.shape[2:])
        frames = frames.float() / 255.0
        frames = (frames - self.mean) / self.std
        features = self.encoder(frames)
        features = features.view(B, T, -1)
        pooled = features.mean(dim=1)
        return self.classifier(pooled) / self.temperature


def load_model(weights_path, num_classes=2):
    model = VideoDeepfakeDetector(num_classes=num_classes)
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    model.train(False)
    return model
''')

# Create zip
zip_path = '/content/video_detector_v3.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(out / 'model.safetensors', 'model.safetensors')
    zf.write(out / 'model_config.yaml', 'model_config.yaml')
    zf.write(out / 'model.py', 'model.py')

size_mb = os.path.getsize(zip_path) / 1e6
print(f'\nDone! Output: {zip_path} ({size_mb:.1f} MB)')
print(f'Final sn34: {final["sn34_score"]:.4f}')
print(f'\nDownload the zip, then on Ubuntu run:')
print(f'  cd ~/bitmind-subnet && source .venv/bin/activate')
print(f'  python push_standalone.py --video-model video_detector_v3.zip --wallet-name miner')

In [ ]:
# Download the zip file
from google.colab import files
files.download('/content/video_detector_v3.zip')